# Production Patterns for Embeddings and Vector Stores

Notebooks prove retrieval works; production requires **versioned models**, **idempotent ingest**, **observability**, **cost controls**, and **tenant isolation** — the same discipline as prompt management (**04. Prompt Engineering → 10**).

This notebook closes **06. Embeddings & Vector DBs**: versioning artifacts, batch and incremental ingest, structured logging, security, multi-tenancy, re-indexing, health checks, and how this section wires into **08. RAG Fundamentals**.

Prerequisites: **01–09** in this section.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import hashlib
import json
import time
import uuid
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"
BATCH_SIZE = 50

---

## 1. Production vs Notebook

```
Notebook                         Production
────────                         ──────────
Re-run all cells                 Scheduled ingest jobs
Ephemeral Chroma client          Persistent path + backups
Print top-3 ids                  Structured logs + metrics
Hard-coded model name            Versioned config + env vars
Manual eyeball eval              Recall@k CI gate (notebook 09)
```

| Concern | Notebook habit | Production pattern |
|---------|------------------|-------------------|
| **Reproducibility** | Re-embed ad hoc | Pin model + chunk version |
| **Failures** | Retry manually | Backoff, checkpoints, alerts |
| **Scale** | 10 chunks | Millions — batch + incremental |
| **Ownership** | One author | Runbooks, on-call, SLIs |

---

## 2. Version Everything

Every search request must be traceable to the **index build** that produced it.

| Artifact | Version as | Example |
|----------|------------|--------|
| Embedding model | env / config | `text-embedding-3-small` |
| Chunk strategy | slug | `chunk_v2_512tok_20pct_overlap` |
| Collection | name suffix | `docs_prod_v3` |
| Index build | build id | `2025-07-03T10:00:00Z_a3f9` |

**Golden rule:** never query a collection embedded with model A using query vectors from model B — distances are meaningless.

Store versions in collection metadata (Chroma) or a `index_manifest` table (Postgres/pgvector).

In [ ]:
@dataclass
class IndexManifest:
    collection: str
    embedding_model: str
    chunk_strategy: str
    build_id: str = field(default_factory=lambda: datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"))
    chunk_count: int = 0

    def to_metadata(self) -> dict[str, str]:
        return {k: str(v) for k, v in asdict(self).items()}


manifest = IndexManifest(
    collection="notes_docs_v1",
    embedding_model=EMBED_MODEL,
    chunk_strategy="markdown_512_20overlap",
    chunk_count=0,
)
print(json.dumps(manifest.to_metadata(), indent=2))

---

## 3. Ingest Pipeline Architecture

```
Sources (git, S3, CMS)
    │ extract text
    ▼
Normalize + hash source
    │ chunk (strategy from manifest)
    ▼
Stable chunk ids  ──► skip if content hash unchanged
    │ batch embed
    ▼
Upsert vector DB + metadata
    │ delete stale ids
    ▼
Verify count + sample query smoke test
```

| Practice | Why |
|----------|-----|
| **Stable ids** | `{source_path}#{chunk_index}` or content hash |
| **Content hash** | Skip re-embed when text unchanged |
| **Batch embed** | Fewer API calls, lower cost |
| **Idempotent upsert** | Safe to re-run job |
| **Delete stale** | Source removed → remove orphan chunks |
| **Checkpoint** | Resume after failure mid-corpus |
| **Backoff on 429** | Survive rate limits |

---

## 4. Stable Chunk IDs and Content Hashing

In [ ]:
def content_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


def chunk_id(source: str, index: int) -> str:
    return f"{source}#{index}"


SAMPLE_CHUNKS = [
    {"source": "db-docs/alembic.md", "index": 0, "text": "Alembic applies SQLAlchemy schema migrations."},
    {"source": "security/jwt.md", "index": 0, "text": "JWT bearer tokens authenticate API requests."},
    {"source": "api/fastapi.md", "index": 0, "text": "Notes API is built with FastAPI."},
]

for c in SAMPLE_CHUNKS:
    cid = chunk_id(c["source"], c["index"])
    ch = content_hash(c["text"])
    print(f"{cid:30s}  hash={ch}")

On re-ingest, compare stored `content_hash` in metadata — if unchanged, skip the embed API call.

---

## 5. Batch Embedding with Retry

In [ ]:
def batch_embed(
    texts: list[str],
    batch_size: int = BATCH_SIZE,
    max_retries: int = 3,
    pause_sec: float = 0.1,
) -> list[list[float]]:
    all_embeddings: list[list[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        for attempt in range(max_retries):
            try:
                response = client.embeddings.create(model=EMBED_MODEL, input=batch)
                all_embeddings.extend([d.embedding for d in response.data])
                break
            except Exception as exc:
                if attempt == max_retries - 1:
                    raise
                wait = 2 ** attempt
                print(f"Batch {i // batch_size} failed ({exc}); retry in {wait}s")
                time.sleep(wait)
        time.sleep(pause_sec)
    return all_embeddings


chunks = [f"Synthetic doc chunk {i} about FastAPI, Alembic, JWT, or pytest." for i in range(120)]
vectors = batch_embed(chunks, batch_size=40)

print(f"Embedded {len(vectors)} chunks in {(len(chunks) + 39) // 40} API batches")
print(f"Vector dimensions: {len(vectors[0])}")

Production jobs log batch index, elapsed time, and token estimates. Persist checkpoint after each successful batch.

---

## 6. Incremental Ingest

Full re-index is expensive. **Incremental** ingest embeds only new or changed chunks.

In [ ]:
@dataclass
class ChunkRecord:
    id: str
    text: str
    content_hash: str
    metadata: dict[str, Any]


def plan_ingest(incoming: list[ChunkRecord], indexed_hashes: dict[str, str]) -> tuple[list[ChunkRecord], list[str]]:
    """Return (to_upsert, ids_to_delete)."""
    to_upsert = []
    incoming_ids = set()
    for chunk in incoming:
        incoming_ids.add(chunk.id)
        if indexed_hashes.get(chunk.id) != chunk.content_hash:
            to_upsert.append(chunk)
    stale_ids = [cid for cid in indexed_hashes if cid not in incoming_ids]
    return to_upsert, stale_ids


indexed = {
    "db-docs/alembic.md#0": content_hash("Alembic applies SQLAlchemy schema migrations."),
    "api/old-page.md#0": content_hash("Deprecated onboarding page."),
}

incoming = [
    ChunkRecord(
        id="db-docs/alembic.md#0",
        text="Alembic applies SQLAlchemy schema migrations.",
        content_hash=content_hash("Alembic applies SQLAlchemy schema migrations."),
        metadata={"source": "db-docs"},
    ),
    ChunkRecord(
        id="security/jwt.md#0",
        text="JWT bearer tokens authenticate API requests.",
        content_hash=content_hash("JWT bearer tokens authenticate API requests."),
        metadata={"source": "security-docs"},
    ),
]

to_upsert, stale = plan_ingest(incoming, indexed)
print(f"Upsert {len(to_upsert)} changed/new chunks (skip unchanged)")
print(f"Delete {len(stale)} stale ids:", stale)
print("IDs to embed:", [c.id for c in to_upsert])

Only embed `to_upsert` — unchanged Alembic chunk is skipped; deleted `api/old-page.md` is removed from the index.

---

## 7. Observability

### 7.1 Log Every Retrieval

Structured logs enable debugging, cost attribution, and quality regression detection.

```json
{
  "event": "retrieval",
  "request_id": "550e8400-e29b-41d4-a716-446655440000",
  "collection": "notes_docs_v1",
  "embedding_model": "text-embedding-3-small",
  "chunk_strategy": "markdown_512_20overlap",
  "top_k": 5,
  "latency_ms": 42,
  "filter": {"tenant_id": "acme"},
  "top_scores": [0.82, 0.79, 0.71],
  "chunk_ids": ["db-docs/alembic.md#0", "security/jwt.md#0"]
}
```

### 7.2 Metrics and Alerts

| Signal | Alert when |
|--------|------------|
| `retrieval_latency_p99` | Spikes above SLO |
| `empty_results_rate` | Sudden increase |
| `top_score_mean` | Collapses (bad index / wrong model) |
| `ingest_lag_seconds` | Source updated but index stale |
| `embed_api_errors` | Sustained 429/5xx |

In [ ]:
@dataclass
class RetrievalEvent:
    request_id: str
    collection: str
    embedding_model: str
    top_k: int
    latency_ms: float
    chunk_ids: list[str]
    top_scores: list[float]
    filter: dict[str, Any] | None = None
    chunk_strategy: str = ""

    def to_log_line(self) -> str:
        payload = {"event": "retrieval", **asdict(self)}
        return json.dumps(payload)


event = RetrievalEvent(
    request_id=str(uuid.uuid4()),
    collection="notes_docs_v1",
    embedding_model=EMBED_MODEL,
    chunk_strategy="markdown_512_20overlap",
    top_k=5,
    latency_ms=38.4,
    chunk_ids=["db-docs/alembic.md#0", "security/jwt.md#0"],
    top_scores=[0.84, 0.79],
    filter={"tenant_id": "acme"},
)
print(event.to_log_line())

---

## 8. Query Embedding Cache

Repeat queries (support portals, canned FAQs) should not re-hit the embed API.

In [ ]:
class QueryEmbedCache:
    def __init__(self, model: str):
        self.model = model
        self._cache: dict[str, list[float]] = {}
        self.hits = 0
        self.misses = 0

    def _key(self, text: str) -> str:
        return f"{self.model}:{content_hash(text)}"

    def embed(self, text: str, client: OpenAI) -> list[float]:
        key = self._key(text)
        if key in self._cache:
            self.hits += 1
            return self._cache[key]
        self.misses += 1
        vec = client.embeddings.create(model=self.model, input=[text]).data[0].embedding
        self._cache[key] = vec
        return vec


cache = QueryEmbedCache(EMBED_MODEL)
q = "How do I run Alembic migrations?"
v1 = cache.embed(q, client)
v2 = cache.embed(q, client)
print(f"Cache hits={cache.hits} misses={cache.misses} same_vector={v1 == v2}")

Production: use Redis with TTL, keyed by `(model, normalized_query_hash)`. Invalidate on model version change.

---

## 9. Security and Privacy

| Risk | Mitigation |
|------|------------|
| Secrets in source docs | Scan + block before embed |
| PII in chunks | Redact at ingest; policy per tenant |
| Cross-tenant leakage | Metadata filter + separate collections |
| Embedding inversion | Treat vectors like source text sensitivity |
| Unauthorized query | Auth on RAG API; audit logs |

Embeddings are **not encryption** — assume an attacker with index access can infer approximate content. Encrypt disks, restrict network access, and scope API keys.

**Never** embed API keys, passwords, or private keys — even "internal" indexes get copied to dev laptops.

---

## 10. Multi-Tenancy Patterns

| Pattern | Isolation | Complexity |
|---------|-----------|------------|
| **Single collection + `tenant_id` filter** | Logical | Low; filter bugs are risky |
| **Collection per tenant** | Stronger | Medium; many collections |
| **Database per tenant** | Strongest | High ops |

Always include `tenant_id` in retrieval logs. Integration tests should assert tenant A cannot retrieve tenant B chunks (notebook 07 filters).

---

## 11. Cost Controls

| Lever | Ingest | Query | Storage |
|-------|--------|-------|--------|
| Smaller model (`3-small` vs `3-large`) | ↓ | ↓ | — |
| Reduced `dimensions` | ↓ | ↓ | ↓ |
| Incremental ingest | ↓ | — | — |
| Query embed cache | — | ↓ | small RAM |
| Right-size chunks | ↓ vectors | — | ↓ |
| Right-size `top_k` | — | — | ↓ LLM tokens |

Track **$/1M embed tokens** and **$/GB vector storage** monthly. Largest surprise is re-embedding entire corpus after a chunk strategy change without incremental hashing.

---

## 12. Re-Indexing and Blue/Green

Full re-index when (see notebook 07):

- Embedding **model** changes
- **Chunk strategy** changes
- **Distance metric** changes

**Blue/green flow:**

```
1. Build notes_docs_v2 (green) alongside notes_docs_v1 (blue)
2. Run Recall@k on green (notebook 09)
3. Feature-flag 10% query traffic to green
4. Promote green → blue; retire old collection
```

Keep v1 until rollback window closes — deleting the only copy before validation is a common outage.

---

## 13. Health Checks and SLIs

| Check | Purpose |
|-------|--------|
| `collection.count()` vs expected | Detect incomplete ingest |
| Canary query with known top hit | Detect wrong model / empty index |
| Embed API latency | Upstream dependency |
| Disk usage (Chroma path) | Capacity planning |

**SLI example:** 99% of retrieval requests return ≥1 result with top score > 0.5 on canary queries (domain-tuned threshold).

---

## 14. Production Checklist

```
□ EMBEDDING_MODEL and CHUNK_STRATEGY in version control
□ IndexManifest logged on every build
□ Stable chunk ids + content_hash incremental ingest
□ Batch embed with retry/backoff
□ Stale chunk deletion on source removal
□ Retrieval structured logs (request_id, model, collection, ids, scores)
□ Recall@k eval before collection promotion (notebook 09)
□ Tenant isolation tested
□ No secrets in corpus
□ Blue/green path for model/chunk migrations
□ Runbook: re-index, rollback, on-call alerts
```

---

## 15. Connection to RAG Fundamentals

This section built the retrieval layer:

| Step | Notebook | Output |
|------|----------|--------|
| Embeddings | 01–03 | Vectors from text |
| Chunking | 04 | Retrievable units |
| Vector DB | 05–07 | Store, filter, manage |
| Vendor choice | 08 | Architecture decision |
| Quality | 09 | Recall@k, MRR |
| Operations | 10 | Ingest, logs, cost |

**08. RAG Fundamentals** wires retrieval into generation:

- `01_naive_rag.py` — minimal pipeline
- `02_rag_with_openai.py` — API-backed answers
- `knowledge_base.py` — shared corpus helpers

Refactor those scripts to use your `IndexManifest`, `batch_embed`, and `VectorStore` adapter (notebook 08) instead of ad-hoc notebook code.

---

## 16. Section Recap

| # | Notebook | Topic |
|---|----------|-------|
| 01 | Introduction to Embeddings | Dense vectors, similarity intuition |
| 02 | Similarity Search | Cosine, L2, ANN |
| 03 | Embedding APIs | Batching, dimensions, model choice |
| 04 | Document Chunking | Strategies, overlap, metadata |
| 05 | Introduction to Vector DBs | Data model, in-memory store |
| 06 | Chroma | Collections, ingest, query |
| 07 | Metadata Filters | `where`, upsert, blue/green |
| 08 | Landscape | Trade-offs, hybrid, portability |
| 09 | Evaluating Retrieval | Recall@k, MRR, A/B |
| 10 | Production Patterns | Versioning, ingest, observability |

---

## 17. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Query model ≠ index model | Random rankings | IndexManifest + validation |
| Full re-embed every night | Cost explosion | Content-hash incremental |
| No stale delete | Wrong outdated answers | Diff ingest plan |
| Logs without chunk ids | Can't debug RAG | Structured RetrievalEvent |
| Skip eval before promotion | Production regression | Recall@k gate |
| Shared index for all tenants | Data leak | Filter + integration tests |
| One giant collection forever | Painful migrations | Versioned collection names |

---

## 18. Summary

Operate embeddings and vector stores like production software: **version** models and collections, **batch** with retry, **incremental** ingest via content hashes, **log** every retrieval, **eval** before promotion, and **isolate** tenants.

This completes **06. Embeddings & Vector DBs**. Continue with **08. RAG Fundamentals** to connect retrieval to LLM generation.